# Bonn EEG ML: Final Clean Pipeline

Traditional ML only. Includes three-class classification, binary seizure detection, focused S vs N+F experiment, simple plots, model export, and integrity checks.


In [1]:
# 1. Imports and configuration
from pathlib import Path
import os, json, time, warnings, platform, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from scipy.signal import butter, sosfiltfilt, welch
from scipy.stats import skew, kurtosis

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score, balanced_accuracy_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="scipy")

RANDOM_STATE = 42
FS = 173.61
LOWCUT, HIGHCUT = 0.5, 40.0
FILTER_ORDER = 4
EPOCH_SECONDS = None  # Bonn raw files are processed as full recordings, not 5-second windows
EPOCH_SAMPLES = int(FS * EPOCH_SECONDS)
BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "gamma": (30.0, 40.0),
}
LABEL_NAMES = {0: "Normal EEG", 1: "Seizure-free", 2: "Active seizure"}
POSITIVE_SEIZURE_LABEL = 2
CODE_TO_LABEL = {"Z": 0, "O": 0, "N": 1, "F": 1, "S": 2}

BASE_DIR = Path.cwd()
# Also support notebook execution from /mnt/data in this environment.
if not (BASE_DIR / "bonn_preprocessed_features.csv").exists() and Path("/mnt/data/bonn_preprocessed_features.csv").exists():
    BASE_DIR = Path("/mnt/data")

DATASET_DIR = BASE_DIR / "bonn_dataset"
OUTPUT_DIR = BASE_DIR / "bonn_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Working directory:", BASE_DIR)
print("Raw dataset directory exists:", DATASET_DIR.exists())
print("Output directory:", OUTPUT_DIR.resolve())

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

## 1. Load and audit the dataset

In [ ]:
# 2. Load and audit feature data
FEATURE_PATH = BASE_DIR / "bonn_preprocessed_features.csv"
TRAIN_PATH = BASE_DIR / "bonn_train_features.csv"
TEST_PATH = BASE_DIR / "bonn_test_features.csv"

if FEATURE_PATH.exists():
    data = pd.read_csv(FEATURE_PATH)
    DATA_SOURCE_MODE = "precomputed_features"
elif TRAIN_PATH.exists() and TEST_PATH.exists():
    data = pd.concat([pd.read_csv(TRAIN_PATH), pd.read_csv(TEST_PATH)], ignore_index=True)
    DATA_SOURCE_MODE = "combined_train_test_features"
else:
    data = None
    DATA_SOURCE_MODE = "raw_dataset_required"

if data is not None:
    if "label" not in data.columns:
        if "code" not in data.columns:
            raise ValueError("Feature CSV needs a 'label' or 'code' column.")
        data["label"] = data["code"].map(CODE_TO_LABEL)
    if data["label"].isna().any():
        raise ValueError("Unrecognized or missing target labels detected.")
    data["label"] = data["label"].astype(int)
    if "file" not in data.columns:
        data["file"] = [f"unknown_recording_{i}" for i in range(len(data))]
        print("WARNING: no recording ID found. Leakage-safe grouped splitting is not possible.")
    metadata_candidates = {
        "file", "path", "code", "set", "medical_meaning", "condition",
        "label", "epoch_id", "subject", "subject_id", "patient", "patient_id",
        "sample_count", "duration_seconds"
    }
    feature_cols = [
        c for c in data.columns
        if c not in metadata_candidates and pd.api.types.is_numeric_dtype(data[c])
    ]
    if not feature_cols:
        raise ValueError("No numeric feature columns were detected.")
    print("Source mode:", DATA_SOURCE_MODE)
    print("Rows / epochs:", len(data))
    print("Unique recordings:", data["file"].nunique())
    print("Numeric features:", len(feature_cols))
    print("Class distribution (epochs):")
    display(data["label"].map(LABEL_NAMES).value_counts().rename_axis("class").to_frame("epochs"))
    print("Class distribution (recordings):")
    display(data.drop_duplicates("file")["label"].map(LABEL_NAMES).value_counts().rename_axis("class").to_frame("recordings"))
    print("Duplicate complete rows:", int(data.duplicated().sum()))
    print("Non-finite numeric feature values:", int((~np.isfinite(data[feature_cols].to_numpy(dtype=float))).sum()))
    if "code" in data.columns:
        print("Bonn code distribution:")
        display(data["code"].value_counts(dropna=False).to_frame("recordings_or_epochs"))
else:
    print("No precomputed feature CSV found. Raw feature extraction mode will be used if bonn_dataset/ exists.")

## 2. Raw EEG preprocessing and feature extraction

### Raw EEG helper functions


In [ ]:
def load_raw_bonn_file(path):
    path = Path(path)
    try:
        arr = np.loadtxt(path, dtype=float)
    except Exception as exc:
        raise ValueError(f"Could not parse EEG file {path}: {exc}") from exc
    arr = np.asarray(arr, dtype=float).reshape(-1)
    if arr.size == 0:
        raise ValueError(f"Empty EEG recording: {path}")
    if not np.isfinite(arr).all():
        raise ValueError(f"NaN/Inf values found in EEG recording: {path}")
    if arr.size < 2:
        raise ValueError(f"EEG recording too short: {path}")
    return arr


In [ ]:
def detect_bonn_code(path):
    p = Path(path)
    for part in reversed(p.parts):
        code = part.upper()
        if code in CODE_TO_LABEL:
            return code
    stem = p.stem.upper()
    for code in ("Z", "O", "N", "F", "S"):
        if stem.startswith(code):
            return code
    return None


In [ ]:
def bandpass_filter(signal, fs=FS, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER):
    x = np.asarray(signal, dtype=float)
    if x.ndim != 1 or x.size < max(30, 3 * (2 * order + 1)):
        raise ValueError("Signal must be 1-D and long enough for filtering.")
    if not np.isfinite(x).all():
        raise ValueError("Signal contains NaN or infinite values.")
    if not (0 < low < high < fs / 2):
        raise ValueError(f"Invalid filter/frequency configuration: low={low}, high={high}, fs={fs}")
    sos = butter(order, [low, high], btype="bandpass", fs=fs, output="sos")
    try:
        return sosfiltfilt(sos, x)
    except ValueError as exc:
        raise ValueError(f"Signal too short for zero-phase filtering: {exc}") from exc


In [ ]:
def segment_signal(signal, epoch_samples=EPOCH_SAMPLES):
    x = np.asarray(signal, dtype=float)
    if x.ndim != 1 or x.size == 0:
        raise ValueError("Expected a non-empty one-dimensional signal.")
    if not np.isfinite(x).all():
        raise ValueError("Signal contains NaN or infinite values.")
    if epoch_samples <= 0:
        raise ValueError("epoch_samples must be positive.")
    n_epochs = x.size // epoch_samples
    if n_epochs == 0:
        raise ValueError(f"Signal shorter than one epoch ({epoch_samples} samples).")
    discarded = int(x.size - n_epochs * epoch_samples)
    epochs = x[:n_epochs * epoch_samples].reshape(n_epochs, epoch_samples)
    return epochs, discarded


In [ ]:
def hjorth_features(x):
    x = np.asarray(x, dtype=float)
    var0 = float(np.var(x))
    dx = np.diff(x)
    ddx = np.diff(dx)
    var1 = float(np.var(dx)) if dx.size else 0.0
    var2 = float(np.var(ddx)) if ddx.size else 0.0
    eps = 1e-12
    activity = var0
    mobility = np.sqrt(max(var1, 0.0) / max(var0, eps))
    complexity = np.sqrt(max(var2, 0.0) / max(var1, eps)) / max(mobility, eps)
    return {"hjorth_activity": activity, "hjorth_mobility": mobility, "hjorth_complexity": complexity}


In [ ]:
def spectral_features(x, fs=FS):
    x = np.asarray(x, dtype=float)
    f, pxx = welch(x, fs=fs, nperseg=min(256, len(x)))
    if len(f) < 2 or not np.isfinite(pxx).all():
        raise ValueError("PSD calculation failed.")
    df = float(f[1] - f[0])
    total_mask = (f >= 0.5) & (f <= min(40.0, fs / 2))
    total_power = float(np.trapezoid(pxx[total_mask], f[total_mask])) if total_mask.sum() >= 2 else 0.0
    total_power = max(total_power, 1e-12)
    out = {}
    for name, (lo, hi) in BANDS.items():
        hi = min(hi, fs / 2)
        mask = (f >= lo) & (f <= hi)
        # Include the sampled band interval; a one-bin band has zero integrated width.
        power = float(np.trapezoid(pxx[mask], f[mask])) if mask.sum() >= 2 else float(pxx[mask][0] * df) if mask.sum() == 1 else 0.0
        out[f"{name}_power"] = power
        out[f"{name}_relative_power"] = power / total_power
    ps = pxx / max(float(np.sum(pxx)), 1e-12)
    ps = ps[ps > 0]
    out["spectral_entropy"] = float(-np.sum(ps * np.log2(ps)))
    out["dominant_frequency"] = float(f[np.argmax(pxx)])
    centroid = float(np.sum(f * pxx) / max(np.sum(pxx), 1e-12))
    out["spectral_centroid"] = centroid
    out["spectral_bandwidth"] = float(np.sqrt(np.sum(((f-centroid)**2) * pxx) / max(np.sum(pxx), 1e-12)))
    out["peak_psd"] = float(np.max(pxx))
    return out


In [ ]:
def wavelet_haar_features(x, max_levels=5):
    cur = np.asarray(x, dtype=float).copy()
    coeffs = []
    for level in range(1, max_levels + 1):
        if len(cur) < 2:
            break
        even, odd = cur[::2], cur[1::2]
        n = min(len(even), len(odd))
        approx = (even[:n] + odd[:n]) / np.sqrt(2)
        detail = (even[:n] - odd[:n]) / np.sqrt(2)
        coeffs.append((level, detail))
        cur = approx
    coeffs.append((len(coeffs) + 1, cur))
    energies = [float(np.sum(c*c)) for _, c in coeffs]
    total = max(sum(energies), 1e-12)
    out = {}
    for (level, _), energy in zip(coeffs, energies):
        out[f"wavelet_L{level}_energy"] = energy
        out[f"wavelet_L{level}_relative_energy"] = energy / total
    return out


In [ ]:
def permutation_entropy(x, order=3, delay=1):
    x = np.asarray(x, dtype=float)
    n = len(x) - delay * (order - 1)
    if n <= 0:
        return 0.0
    counts = {}
    for i in range(n):
        pattern = tuple(np.argsort(x[i:i + delay * order:delay], kind="stable"))
        counts[pattern] = counts.get(pattern, 0) + 1
    probs = np.asarray(list(counts.values()), dtype=float)
    probs /= probs.sum()
    return float(-np.sum(probs * np.log(probs + 1e-12)) / np.log(math.factorial(order)))


In [ ]:
def approximate_entropy(x, m=2, r=None):
    x = np.asarray(x, dtype=float)
    if len(x) <= m + 1:
        return 0.0
    if r is None:
        r = 0.2 * float(np.std(x))
    if r <= 0:
        return 0.0
    def phi(mm):
        emb = np.asarray([x[i:i + mm] for i in range(len(x) - mm + 1)])
        counts = []
        for row in emb:
            distances = np.max(np.abs(emb - row), axis=1)
            counts.append(np.mean(distances <= r))
        return float(np.mean(np.log(np.asarray(counts) + 1e-12)))
    return float(phi(m) - phi(m + 1))


In [ ]:
def extract_epoch_features(x, fs=FS):
    x = np.asarray(x, dtype=float)
    if x.ndim != 1 or len(x) < 16 or not np.isfinite(x).all():
        raise ValueError("Invalid epoch.")
    eps = 1e-12
    mean = float(np.mean(x))
    std = float(np.std(x))
    q25, q75 = np.percentile(x, [25, 75])
    signs = np.signbit(x)
    out = {
        "mean": mean,
        "std": std,
        "variance": float(np.var(x)),
        "rms": float(np.sqrt(np.mean(x*x))),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "range": float(np.ptp(x)),
        "median": float(np.median(x)),
        "iqr": float(q75-q25),
        "skewness": float(skew(x, bias=False)) if std > eps else 0.0,
        "kurtosis": float(kurtosis(x, bias=False)) if std > eps else 0.0,
        "zero_crossing_rate": float(np.mean(signs[1:] != signs[:-1])),
        "line_length": float(np.mean(np.abs(np.diff(x)))),
        "approximate_entropy": approximate_entropy(x),
        "permutation_entropy": permutation_entropy(x),
    }
    out.update(hjorth_features(x))
    out.update(spectral_features(x, fs))
    out.update(wavelet_haar_features(x))
    return {k: float(v) for k, v in out.items()}


In [ ]:
def build_features_from_raw(root=DATASET_DIR):
    """
    Bonn-specific loader: one original ~23.6 s recording = one training example.
    Do not split the original Bonn segment into artificial subwindows by default.
    The canonical sets contain 100 files each for Z/O/N/F/S.
    """
    root = Path(root)
    files = [p for p in root.rglob("*") if p.is_file() and detect_bonn_code(p)]
    if not files:
        raise FileNotFoundError(f"No Bonn Z/O/N/F/S text files found under {root}")
    rows = []
    lengths = []
    for path in sorted(files):
        code = detect_bonn_code(path)
        signal = load_raw_bonn_file(path)
        lengths.append(len(signal))
        filtered = bandpass_filter(signal, fs=FS)
        row = extract_epoch_features(filtered, fs=FS)
        row.update({
            "file": path.name,
            "path": str(path.resolve()),
            "code": code,
            "label": CODE_TO_LABEL[code],
            "condition": LABEL_NAMES[CODE_TO_LABEL[code]],
            "sample_count": int(len(signal)),
            "duration_seconds": float(len(signal) / FS),
        })
        rows.append(row)
    result = pd.DataFrame(rows)
    print(f"Raw recordings loaded: {len(files)}")
    print("Sample counts observed:", pd.Series(lengths).value_counts().to_dict())
    print("One row per original Bonn recording; no artificial windowing applied.")
    if any(n < 1000 for n in lengths):
        raise ValueError("Some raw files are unexpectedly short for the Bonn dataset. Check folder/file parsing.")
    return result


In [ ]:
if data is None and DATASET_DIR.exists():
    data = build_features_from_raw()
    DATA_SOURCE_MODE = "raw_extracted"
    FEATURE_PATH = OUTPUT_DIR / "bonn_preprocessed_features_rebuilt.csv"
    data.to_csv(FEATURE_PATH, index=False)
    metadata_candidates = {"file", "path", "code", "set", "medical_meaning", "condition", "label", "epoch_id", "subject", "subject_id", "patient", "patient_id"}
    feature_cols = [c for c in data.columns if c not in metadata_candidates and pd.api.types.is_numeric_dtype(data[c])]
elif data is None:
    raise FileNotFoundError(
        "No feature CSVs or bonn_dataset/ folder found. Put the supplied feature CSV beside this notebook, "
        "or add bonn_dataset/{Z,O,N,F,S}/ with the raw Bonn text files."
    )


## 3. Recording-grouped train/test split

In [ ]:
# 4. Recording-grouped holdout
GROUP_COLUMN = "file"
if GROUP_COLUMN not in data.columns:
    raise ValueError("Cannot group-split because recording IDs are unavailable.")

data = data.copy()
data = data.replace([np.inf, -np.inf], np.nan)

# Keep only numeric feature columns that have at least one observed value.
metadata_candidates = {"file", "path", "code", "set", "medical_meaning", "condition", "label",
                       "epoch_id", "subject", "subject_id", "patient", "patient_id",
                       "sample_count", "duration_seconds"}
feature_cols = [
    c for c in feature_cols
    if c in data.columns and c not in metadata_candidates and data[c].notna().any()
]
if not feature_cols:
    raise ValueError("No usable numeric features.")

# Ensure each recording has one consistent label.
group_label_counts = data.groupby(GROUP_COLUMN)["label"].nunique()
if (group_label_counts > 1).any():
    bad_groups = group_label_counts[group_label_counts > 1].index.tolist()[:10]
    raise ValueError(f"Recording groups have inconsistent labels: {bad_groups}")

groups = data[GROUP_COLUMN].astype(str).to_numpy()
X = data[feature_cols].astype(float)
y = data["label"].astype(int).to_numpy()

# GroupShuffleSplit preserves recording boundaries. We check class coverage after splitting.
splitter = GroupShuffleSplit(n_splits=100, test_size=0.20, random_state=RANDOM_STATE)
selected = None
for train_idx, test_idx in splitter.split(X, y, groups):
    if set(np.unique(y[train_idx])) == set(np.unique(y)) and set(np.unique(y[test_idx])) == set(np.unique(y)):
        selected = (train_idx, test_idx)
        break
if selected is None:
    raise ValueError("Could not find a grouped split containing all classes. Inspect group/class distribution.")

train_idx, test_idx = selected
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

overlap = set(groups_train) & set(groups_test)
assert not overlap, f"Recording leakage found: {list(overlap)[:5]}"
assert set(np.unique(y_train)) == set(np.unique(y_test)) == set(np.unique(y))

print("Train epochs:", len(train_idx), "Test epochs:", len(test_idx))
print("Train recordings:", len(set(groups_train)), "Test recordings:", len(set(groups_test)))
print("Recording overlap:", len(overlap))
print("Train class counts:", pd.Series(y_train).map(LABEL_NAMES).value_counts().to_dict())
print("Test class counts:", pd.Series(y_test).map(LABEL_NAMES).value_counts().to_dict())

# Save split provenance.
split_manifest = pd.DataFrame({
    "file": list(set(groups_train)) + list(set(groups_test)),
    "split": ["train"] * len(set(groups_train)) + ["test"] * len(set(groups_test))
})
split_manifest.to_csv(OUTPUT_DIR / "recording_split_manifest.csv", index=False)

## 4. Traditional ML models

In [ ]:
# 5. Define pipelines and search spaces
models = {
    "Logistic Regression": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE))
        ]),
        {
            "model__C": [0.1, 1.0, 10.0],
            "model__solver": ["lbfgs"]
        }
    ),
    "SVM (RBF)": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE))
        ]),
        {
            "model__C": [0.1, 1.0, 10.0],
            "model__gamma": ["scale", 0.01, 0.1]
        }
    ),
    "Random Forest": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))
        ]),
        {
            "model__n_estimators": [200, 400],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", 0.7]
        }
    ),
    "Extra Trees": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))
        ]),
        {
            "model__n_estimators": [200, 400],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", 0.7]
        }
    ),
    "Decision Tree": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE))
        ]),
        {
            "model__max_depth": [None, 5, 10, 20],
            "model__min_samples_leaf": [1, 2, 5, 10],
            "model__criterion": ["gini", "entropy"]
        }
    ),
    "KNN": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        {
            "model__n_neighbors": [3, 5, 7, 11],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        }
    ),
    "Gradient Boosting": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
        ]),
        {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.03, 0.1],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [1, 3]
        }
    ),
}

# Use grouped folds. Number of folds cannot exceed the least represented class at group level.
group_meta = data[[GROUP_COLUMN, "label"]].drop_duplicates()
groups_per_class = group_meta.groupby("label")[GROUP_COLUMN].nunique()
n_splits = int(min(5, groups_per_class.min()))
if n_splits < 2:
    raise ValueError("Not enough independent recording groups per class for grouped cross-validation.")
cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
print("Grouped CV folds:", n_splits)
print("Groups per class:", groups_per_class.to_dict())

## 5. Three-class model training and evaluation

In [ ]:
# 6. Fit each model with grouped CV and evaluate once on held-out recordings
results = []
fitted_models = {}
predictions = {}
probabilities = {}
confusion_matrices = {}
reports = {}

for model_name, (pipeline, param_dist) in models.items():
    print(f"\nTraining: {model_name}")
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_dist,
        n_iter=min(10, int(np.prod([len(v) for v in param_dist.values()]))),
        scoring="f1_macro",
        cv=cv,
        refit=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        error_score="raise",
        return_train_score=False,
    )
    fit_start = time.perf_counter()
    search.fit(X_train, y_train, groups=groups_train)
    training_seconds = time.perf_counter() - fit_start
    best = search.best_estimator_

    infer_start = time.perf_counter()
    pred = best.predict(X_test)
    inference_seconds = (time.perf_counter() - infer_start) / max(len(X_test), 1)

    p = None
    try:
        if hasattr(best, "predict_proba"):
            p = best.predict_proba(X_test)
    except Exception:
        p = None

    cm = confusion_matrix(y_test, pred, labels=[0, 1, 2])
    report = classification_report(
        y_test, pred, labels=[0, 1, 2],
        target_names=[LABEL_NAMES[i] for i in [0, 1, 2]],
        output_dict=True, zero_division=0
    )

    # One-vs-rest macro ROC-AUC for multiclass probabilities, when available.
    auc_macro = np.nan
    if p is not None and p.shape[1] == 3:
        try:
            auc_macro = roc_auc_score(y_test, p, multi_class="ovr", average="macro", labels=best.classes_)
        except Exception:
            pass

    # Treat active seizure as the clinically relevant positive class for one-vs-rest metrics.
    active = POSITIVE_SEIZURE_LABEL
    active_tp = int(cm[active, active])
    active_fn = int(cm[active, :].sum() - active_tp)
    active_fp = int(cm[:, active].sum() - active_tp)
    active_tn = int(cm.sum() - active_tp - active_fn - active_fp)
    active_precision = active_tp / (active_tp + active_fp) if active_tp + active_fp else 0.0
    active_recall = active_tp / (active_tp + active_fn) if active_tp + active_fn else 0.0
    active_specificity = active_tn / (active_tn + active_fp) if active_tn + active_fp else 0.0
    active_f1 = (2 * active_precision * active_recall / (active_precision + active_recall)
                 if active_precision + active_recall else 0.0)

    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "macro_precision": precision_score(y_test, pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_test, pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_test, pred, average="weighted", zero_division=0),
        "active_seizure_precision": active_precision,
        "active_seizure_sensitivity": active_recall,
        "active_seizure_specificity_ovr": active_specificity,
        "active_seizure_f1": active_f1,
        "macro_roc_auc_ovr": auc_macro,
        "training_seconds_including_search": training_seconds,
        "inference_ms_per_epoch": inference_seconds * 1000,
        "best_cv_macro_f1": search.best_score_,
        "best_params": json.dumps(search.best_params_, default=str),
    })
    fitted_models[model_name] = best
    predictions[model_name] = pred
    probabilities[model_name] = p
    confusion_matrices[model_name] = cm
    reports[model_name] = report
    print("Best CV macro-F1:", round(search.best_score_, 4))
    print("Test accuracy:", round(accuracy_score(y_test, pred), 4))
    print("Active-seizure sensitivity:", round(active_recall, 4))

results_df = pd.DataFrame(results).sort_values(
    ["best_cv_macro_f1", "active_seizure_sensitivity", "macro_f1"],
    ascending=False
).reset_index(drop=True)
display(results_df)
results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

## 6. Binary seizure detection

Label S as seizure and Z/O/N/F as non-seizure. This is the primary binary seizure-versus-nonseizure task.


In [ ]:
# Binary target: 1 = ictal seizure (S), 0 = non-seizure (Z/O/N/F)
y_binary = (data["label"].astype(int).to_numpy() == POSITIVE_SEIZURE_LABEL).astype(int)
y_binary_train = y_binary[train_idx]
y_binary_test = y_binary[test_idx]

binary_results = []
binary_models = {}
binary_predictions = {}
binary_probabilities = {}
binary_confusion_matrices = {}

for model_name, (pipeline, param_dist) in models.items():
    print(f"Training binary model: {model_name}")
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_dist,
        n_iter=min(8, int(np.prod([len(v) for v in param_dist.values()]))),
        scoring="f1",
        cv=cv,
        refit=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        error_score="raise",
        return_train_score=False,
    )
    start = time.perf_counter()
    search.fit(X_train, y_binary_train, groups=groups_train)
    fit_seconds = time.perf_counter() - start
    fitted = search.best_estimator_

    infer_start = time.perf_counter()
    pred = fitted.predict(X_test)
    infer_ms = 1000 * (time.perf_counter() - infer_start) / max(len(X_test), 1)

    probs = None
    if hasattr(fitted, "predict_proba"):
        try:
            probs = fitted.predict_proba(X_test)[:, list(fitted.classes_).index(1)]
        except Exception:
            probs = None

    tn, fp, fn, tp = confusion_matrix(y_binary_test, pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0
    auc = np.nan
    if probs is not None and len(np.unique(y_binary_test)) == 2:
        auc = roc_auc_score(y_binary_test, probs)

    binary_results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_binary_test, pred),
        "precision_seizure": precision_score(y_binary_test, pred, pos_label=1, zero_division=0),
        "seizure_sensitivity_recall": recall_score(y_binary_test, pred, pos_label=1, zero_division=0),
        "specificity_non_seizure": specificity,
        "seizure_f1": f1_score(y_binary_test, pred, pos_label=1, zero_division=0),
        "macro_f1": f1_score(y_binary_test, pred, average="macro", zero_division=0),
        "roc_auc": auc,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "training_seconds_including_search": fit_seconds,
        "inference_ms_per_recording": infer_ms,
        "best_cv_seizure_f1": search.best_score_,
        "best_params": json.dumps(search.best_params_, default=str),
    })
    binary_models[model_name] = fitted
    binary_predictions[model_name] = pred
    binary_probabilities[model_name] = probs
    binary_confusion_matrices[model_name] = np.array([[tn, fp], [fn, tp]])
    print(f"  sensitivity={sensitivity:.3f}, specificity={specificity:.3f}, F1={f1_score(y_binary_test, pred):.3f}")

binary_results_df = pd.DataFrame(binary_results).sort_values(
    ["best_cv_seizure_f1", "seizure_sensitivity_recall", "specificity_non_seizure"],
    ascending=False
).reset_index(drop=True)
display(binary_results_df)
binary_results_df.to_csv(OUTPUT_DIR / "binary_seizure_model_comparison.csv", index=False)

# Confusion matrices and class-wise reports for binary task.
binary_cm_dir = OUTPUT_DIR / "binary_confusion_matrices"
binary_cm_dir.mkdir(exist_ok=True)
for model_name, cm in binary_confusion_matrices.items():
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(cm, display_labels=["Non-seizure", "Seizure"])
    disp.plot(ax=ax, cmap="viridis", colorbar=False, values_format="d")
    ax.set_title(f"{model_name}: Binary Seizure Detection")
    fig.tight_layout()
    safe = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    fig.savefig(binary_cm_dir / f"{safe}_confusion_matrix.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    report = classification_report(
        y_binary_test, binary_predictions[model_name],
        labels=[0, 1], target_names=["Non-seizure", "Seizure"],
        output_dict=True, zero_division=0
    )
    pd.DataFrame(report).T.to_csv(OUTPUT_DIR / f"binary_{safe}_classification_report.csv")

# Select binary model from grouped-CV seizure F1, not test accuracy.
best_binary_name = binary_results_df.iloc[0]["model"]
best_binary_model = binary_models[best_binary_name]
binary_bundle = {
    "pipeline": best_binary_model,
    "model_name": best_binary_name,
    "feature_columns": list(feature_cols),
    "label_names": {0: "Non-seizure", 1: "Seizure"},
    "positive_seizure_label": 1,
    "sampling_frequency_hz": FS,
    "filter": {"type": "Butterworth SOS bandpass", "low_hz": LOWCUT, "high_hz": HIGHCUT, "order": FILTER_ORDER},
    "dataset_mode": DATA_SOURCE_MODE,
    "group_column": GROUP_COLUMN,
    "random_state": RANDOM_STATE,
    "training_recordings": sorted(set(groups_train)),
    "test_recordings": sorted(set(groups_test)),
    "task": "binary seizure vs non-seizure",
    "warning": "Research model only; recording-wise split does not establish patient-independent generalization."
}
BINARY_MODEL_PATH = OUTPUT_DIR / "bonn_eeg_binary_seizure_pipeline.joblib"
joblib.dump(binary_bundle, BINARY_MODEL_PATH)
print("Selected binary seizure model:", best_binary_name)
print("Saved binary model:", BINARY_MODEL_PATH.resolve())


## Additional experiment: S vs N + F

This compares active seizure recordings (S) against N/F seizure-free or interictal recordings, avoiding the scalp-versus-intracranial comparison that affects the full three-class task. This is a Bonn benchmark experiment, not clinical validation.


In [ ]:
# S vs N+F only; exclude Z/O because they are scalp recordings.
code_values = data.get("code", pd.Series(index=data.index, dtype=object)).astype(str).str.upper()
if code_values.isin(["S", "N", "F"]).all() and set(code_values.unique()) == {"S", "N", "F"}:
    subset_mask = code_values.isin(["S", "N", "F"])
    X_snf = data.loc[subset_mask, feature_cols].astype(float)
    y_snf = (code_values.loc[subset_mask] == "S").astype(int).to_numpy()
    groups_snf = data.loc[subset_mask, GROUP_COLUMN].astype(str).to_numpy()
    # Hold out recordings; select a representative model using grouped CV on training only.
    snf_split = GroupShuffleSplit(n_splits=100, test_size=0.20, random_state=RANDOM_STATE)
    chosen = None
    for tr, te in snf_split.split(X_snf, y_snf, groups_snf):
        if len(np.unique(y_snf[tr])) == 2 and len(np.unique(y_snf[te])) == 2:
            chosen = (tr, te); break
    if chosen is None:
        print("S vs N+F skipped: could not create a holdout containing both classes.")
    else:
        tr, te = chosen
        snf_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE))
        ])
        snf_pipeline.fit(X_snf.iloc[tr], y_snf[tr])
        snf_pred = snf_pipeline.predict(X_snf.iloc[te])
        snf_prob = snf_pipeline.predict_proba(X_snf.iloc[te])[:, list(snf_pipeline.classes_).index(1)]
        snf_cm = confusion_matrix(y_snf[te], snf_pred, labels=[0, 1])
        snf_report = classification_report(y_snf[te], snf_pred, labels=[0, 1], target_names=["N/F", "S: seizure"], output_dict=True, zero_division=0)
        pd.DataFrame(snf_report).T.to_csv(OUTPUT_DIR / "S_vs_NF_classification_report.csv")
        pd.DataFrame(snf_cm, index=["Actual N/F", "Actual S"], columns=["Predicted N/F", "Predicted S"]).to_csv(OUTPUT_DIR / "S_vs_NF_confusion_matrix.csv")
        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay(snf_cm, display_labels=["N/F", "S: seizure"]).plot(ax=ax, colorbar=False, values_format="d")
        ax.set_title("S vs N+F: Held-out recording results")
        fig.tight_layout(); fig.savefig(OUTPUT_DIR / "S_vs_NF_confusion_matrix.png", dpi=180, bbox_inches="tight"); plt.show(); plt.close(fig)
        snf_summary = {"accuracy": float(accuracy_score(y_snf[te], snf_pred)), "precision_seizure": float(precision_score(y_snf[te], snf_pred, zero_division=0)), "sensitivity_seizure": float(recall_score(y_snf[te], snf_pred, zero_division=0)), "f1_seizure": float(f1_score(y_snf[te], snf_pred, zero_division=0)), "roc_auc": float(roc_auc_score(y_snf[te], snf_prob)), "train_recordings": int(len(set(groups_snf[tr]))), "test_recordings": int(len(set(groups_snf[te])))}
        with open(OUTPUT_DIR / "S_vs_NF_summary.json", "w") as f: json.dump(snf_summary, f, indent=2)
        print("S vs N+F holdout summary:", json.dumps(snf_summary, indent=2))
else:
    print("S vs N+F experiment skipped: class-code column with S, N and F is required. Do not infer this task from collapsed 3-class labels.")


## Repeated cross-validation and uncertainty

This is a separate robustness check on the three-class task. The mean and 95% interval describe fold scores, not patient-independent generalization. Because Bonn subject IDs are not available in the usual release, folds cannot guarantee patient-independent testing.


In [ ]:
# Repeated stratified CV on the training partition; report mean and a 95% t interval.
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from scipy.stats import t as student_t
cv_rows = []
repeated_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
for model_name, (pipeline, _) in models.items():
    scores = cross_val_score(clone(pipeline), X_train, y_train, cv=repeated_cv, scoring="f1_macro", n_jobs=-1)
    mean_score = float(np.mean(scores)); sd_score = float(np.std(scores, ddof=1))
    margin = float(student_t.ppf(0.975, df=len(scores)-1) * sd_score / np.sqrt(len(scores)))
    cv_rows.append({"model": model_name, "repeated_cv_macro_f1_mean": mean_score, "std": sd_score, "ci95_low": max(0.0, mean_score-margin), "ci95_high": min(1.0, mean_score+margin), "fold_scores": json.dumps(scores.tolist())})
repeated_cv_df = pd.DataFrame(cv_rows).sort_values("repeated_cv_macro_f1_mean", ascending=False)
display(repeated_cv_df.drop(columns="fold_scores"))
repeated_cv_df.to_csv(OUTPUT_DIR / "repeated_cv_confidence_intervals.csv", index=False)
print("These intervals summarize repeated fold variability; they do not correct for subject overlap or establish clinical validity.")


## 7. Confusion matrices and classification reports

In [ ]:
# Save confusion matrices and per-class reports
cm_dir = OUTPUT_DIR / "confusion_matrices"
cm_dir.mkdir(exist_ok=True)

for model_name, cm in confusion_matrices.items():
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(cm, display_labels=[LABEL_NAMES[i] for i in [0, 1, 2]])
    disp.plot(ax=ax, cmap="viridis", colorbar=False, values_format="d")
    ax.set_title(f"{model_name} | held-out recording split")
    plt.tight_layout()
    safe_name = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    fig.savefig(cm_dir / f"{safe_name}_confusion_matrix.png", dpi=180)
    plt.show()
    plt.close(fig)

    report_df = pd.DataFrame(reports[model_name]).T
    report_df.to_csv(OUTPUT_DIR / f"{safe_name}_classification_report.csv")

print("Saved confusion matrices to:", cm_dir.resolve())

## 8. Select and save the three-class model

In [ ]:
# Select and save the three-class model
best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
best_pred = predictions[best_name]
best_prob = probabilities[best_name]

print("Selected model:", best_name)
print("Selection rationale: highest grouped-CV macro F1, with active-seizure sensitivity and held-out macro F1 as tie-breakers.")
print(classification_report(
    y_test, best_pred, labels=[0, 1, 2],
    target_names=[LABEL_NAMES[i] for i in [0, 1, 2]],
    zero_division=0
))

# Persist model plus schema and experimental provenance.
bundle = {
    "pipeline": best_model,
    "model_name": best_name,
    "feature_columns": list(feature_cols),
    "label_names": LABEL_NAMES,
    "positive_seizure_label": POSITIVE_SEIZURE_LABEL,
    "sampling_frequency_hz": FS,
    "filter": {"type": "Butterworth SOS bandpass", "low_hz": LOWCUT, "high_hz": HIGHCUT, "order": FILTER_ORDER},
    "epoch_seconds": EPOCH_SECONDS,
    "epoch_samples": EPOCH_SAMPLES,
    "dataset_mode": DATA_SOURCE_MODE,
    "group_column": GROUP_COLUMN,
    "random_state": RANDOM_STATE,
    "training_recordings": sorted(set(groups_train)),
    "test_recordings": sorted(set(groups_test)),
    "warning": "Research model only. Grouping is by recording, not verified patient ID."
}
MODEL_PATH = OUTPUT_DIR / "bonn_eeg_best_ml_pipeline.joblib"
joblib.dump(bundle, MODEL_PATH)
print("Saved:", MODEL_PATH.resolve())

# Also save all fitted models for transparent comparison.
joblib.dump(
    {"models": fitted_models, "feature_columns": list(feature_cols), "label_names": LABEL_NAMES},
    OUTPUT_DIR / "all_fitted_ml_models.joblib"
)

### Easy-to-explain feature graphs

Histograms show class-wise feature distributions. Feature selection uses ANOVA F-score rather than raw variance. These are exploratory plots, not proof of clinical importance.


In [ ]:
PLOT_DIR = OUTPUT_DIR / "feature_analysis"
PLOT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Correlation uses features only, never target or metadata columns.
corr_data = data[feature_cols].replace([np.inf, -np.inf], np.nan)
corr_data = corr_data.loc[:, corr_data.notna().any()]
corr_matrix = corr_data.corr(method="pearson")

fig, ax = plt.subplots(figsize=(max(10, 0.42 * len(corr_matrix.columns)),
                                max(8, 0.38 * len(corr_matrix.columns))))
im = ax.imshow(corr_matrix.to_numpy(), aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(corr_matrix.columns)))
ax.set_yticks(np.arange(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr_matrix.columns, fontsize=7)
ax.set_title("EEG Feature Correlation Matrix (Pearson)")
fig.colorbar(im, ax=ax, label="Pearson correlation")
fig.tight_layout()
fig.savefig(PLOT_DIR / "feature_correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# Save the numerical correlation matrix for inspection.
corr_matrix.to_csv(PLOT_DIR / "feature_correlation_matrix.csv")


In [ ]:
# Select graph features using ANOVA F-score, not raw variance.
from sklearn.feature_selection import f_classif
plot_data = data.copy().replace([np.inf, -np.inf], np.nan)
valid_features = [c for c in feature_cols if c in plot_data.columns and plot_data[c].notna().any()]
X_plot = plot_data[valid_features].replace([np.inf, -np.inf], np.nan)
X_plot = X_plot.fillna(X_plot.median(numeric_only=True))
try:
    f_scores, _ = f_classif(X_plot, plot_data["label"].astype(int))
    f_scores = np.nan_to_num(f_scores, nan=0.0, posinf=0.0, neginf=0.0)
    plot_features = pd.Series(f_scores, index=valid_features).sort_values(ascending=False).head(6).index.tolist()
except Exception:
    plot_features = valid_features[:min(6, len(valid_features))]
print("Features selected for graphs by ANOVA F-score:", plot_features)


In [ ]:
# Simple histograms: show how one EEG feature is distributed across classes.
# Overlapping histograms are easier to explain than violin plots.
for feature in plot_features:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for cls in sorted(LABEL_NAMES):
        vals = plot_data.loc[plot_data["label"] == cls, feature].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
        if len(vals):
            ax.hist(vals, bins=25, alpha=0.45, label=LABEL_NAMES[cls], density=True)
    ax.set_title(f"Distribution of {feature} by EEG Class")
    ax.set_xlabel(feature.replace("_", " ").title())
    ax.set_ylabel("Relative frequency")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"distribution_{feature}.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
# One simple grid of histograms for quick comparison across EEG classes.
ncols = 3
nrows = int(np.ceil(len(plot_features) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.0 * nrows), squeeze=False)
for ax, feature in zip(axes.flat, plot_features):
    for cls in sorted(LABEL_NAMES):
        vals = plot_data.loc[plot_data["label"] == cls, feature].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
        if len(vals):
            ax.hist(vals, bins=20, alpha=0.4, label=LABEL_NAMES[cls], density=True)
    ax.set_title(feature.replace("_", " ").title())
    ax.set_xlabel("Feature value")
    ax.set_ylabel("Relative frequency")
    ax.grid(axis="y", alpha=0.2)
for ax in axes.flat[len(plot_features):]:
    ax.axis("off")
handles, labels = axes.flat[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=min(len(labels), 3), bbox_to_anchor=(0.5, 1.01))
fig.suptitle("EEG Feature Distributions by Class", y=1.04)
fig.tight_layout()
fig.savefig(PLOT_DIR / "feature_extraction_distribution_grid.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# Feature importance map. Use native tree importance if available, plus permutation
# importance for a model-agnostic view. Permutation importance uses held-out test rows
# and is reported for interpretation only; do not tune on it.
from sklearn.inspection import permutation_importance

importance_pipeline = best_model
final_estimator = importance_pipeline.named_steps["model"]
native_importance = None
if hasattr(final_estimator, "feature_importances_"):
    native_importance = pd.Series(
        final_estimator.feature_importances_, index=feature_cols, name="native_importance"
    ).sort_values(ascending=False)
elif hasattr(final_estimator, "coef_"):
    coef = np.asarray(final_estimator.coef_)
    if coef.ndim == 2:
        native_importance = pd.Series(np.mean(np.abs(coef), axis=0), index=feature_cols, name="mean_abs_coefficient")
    else:
        native_importance = pd.Series(np.abs(coef), index=feature_cols, name="abs_coefficient")
    native_importance = native_importance.sort_values(ascending=False)

perm = permutation_importance(
    importance_pipeline,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
perm_importance = pd.DataFrame({
    "feature": feature_cols,
    "permutation_importance_mean": perm.importances_mean,
    "permutation_importance_std": perm.importances_std
}).sort_values("permutation_importance_mean", ascending=False)
perm_importance.to_csv(PLOT_DIR / "permutation_feature_importance.csv", index=False)

top_n = min(20, len(feature_cols))
top_perm = perm_importance.head(top_n).sort_values("permutation_importance_mean", ascending=True)
fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(top_perm))))
ax.barh(top_perm["feature"], top_perm["permutation_importance_mean"],
        xerr=top_perm["permutation_importance_std"], capsize=2)
ax.set_title(f"Top {len(top_perm)} Features by Permutation Importance ({best_name})")
ax.set_xlabel("Decrease in macro-F1 when feature is permuted")
ax.set_ylabel("Feature")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(PLOT_DIR / "feature_importance_permutation.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

if native_importance is not None:
    native_df = native_importance.rename_axis("feature").reset_index()
    native_df.columns = ["feature", native_importance.name]
    native_df.to_csv(PLOT_DIR / "feature_importance_native.csv", index=False)
    top_native = native_df.head(top_n).sort_values(native_importance.name, ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(top_native))))
    ax.barh(top_native["feature"], top_native[native_importance.name])
    ax.set_title(f"Top {len(top_native)} Native Feature Importances ({best_name})")
    ax.set_xlabel(native_importance.name)
    ax.set_ylabel("Feature")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "feature_importance_native.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Feature analysis plots saved in:", PLOT_DIR.resolve())
print("Files include correlation heatmap/matrix, feature distribution plots, and permutation/native importance maps.")


## 9. Feature-based prediction

In [ ]:
# 9. Inference function
def load_model_bundle(path=MODEL_PATH):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Model bundle not found: {path}")
    bundle = joblib.load(path)
    required = {"pipeline", "feature_columns", "label_names", "model_name"}
    missing = required - set(bundle)
    if missing:
        raise ValueError(f"Invalid model bundle. Missing keys: {missing}")
    return bundle

def predict_feature_rows(feature_frame, bundle=None):
    if bundle is None:
        bundle = load_model_bundle()
    if not isinstance(feature_frame, pd.DataFrame):
        raise TypeError("feature_frame must be a pandas DataFrame.")
    if feature_frame.empty:
        raise ValueError("Input feature frame is empty.")
    expected = list(bundle["feature_columns"])
    missing = [c for c in expected if c not in feature_frame.columns]
    if missing:
        raise ValueError(f"Missing required features: {missing}")
    X_in = feature_frame[expected].copy()
    X_in = X_in.apply(pd.to_numeric, errors="coerce")
    arr = X_in.to_numpy(dtype=float)
    if not np.isfinite(arr).all():
        raise ValueError("Input features contain missing, NaN, or infinite values.")
    pipeline = bundle["pipeline"]
    pred = pipeline.predict(X_in)
    output = pd.DataFrame({
        "predicted_label": pred.astype(int),
        "predicted_class": [bundle["label_names"][int(v)] for v in pred],
        "model_name": bundle["model_name"],
        "prediction_timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "input_status": "valid"
    })
    if hasattr(pipeline, "predict_proba"):
        try:
            probs = pipeline.predict_proba(X_in)
            classes = list(pipeline.classes_)
            for j, class_id in enumerate(classes):
                output[f"probability_{bundle['label_names'][int(class_id)].lower().replace(' ', '_')}"] = probs[:, j]
            output["score_interpretation"] = "classifier probability; calibration not independently established"
        except Exception:
            output["score_interpretation"] = "probability unavailable"
    elif hasattr(pipeline, "decision_function"):
        output["decision_score"] = pipeline.decision_function(X_in).tolist()
        output["score_interpretation"] = "decision score, not probability"
    else:
        output["score_interpretation"] = "no probability or decision score exposed"
    return output

# Smoke test: predictions from the saved bundle must match the fitted model.
loaded_bundle = load_model_bundle()
smoke = predict_feature_rows(X_test.iloc[:5].copy(), loaded_bundle)
assert smoke["predicted_label"].tolist() == best_model.predict(X_test.iloc[:5]).astype(int).tolist()
display(smoke)
print("Saved-model prediction consistency test passed.")

## Binary seizure prediction


In [ ]:
def predict_binary_seizure_features(feature_frame, bundle=None):
    """Predict seizure (1) vs non-seizure (0) from a feature DataFrame."""
    if bundle is None:
        if not Path(BINARY_MODEL_PATH).exists():
            raise FileNotFoundError(f"Binary model not found: {BINARY_MODEL_PATH}")
        bundle = joblib.load(BINARY_MODEL_PATH)
    if not isinstance(feature_frame, pd.DataFrame) or feature_frame.empty:
        raise ValueError("Provide a non-empty pandas DataFrame.")
    expected = list(bundle["feature_columns"])
    missing = [col for col in expected if col not in feature_frame.columns]
    if missing:
        raise ValueError(f"Missing required features: {missing}")
    X_in = feature_frame[expected].apply(pd.to_numeric, errors="coerce")
    if not np.isfinite(X_in.to_numpy(dtype=float)).all():
        raise ValueError("Feature input contains missing, NaN, or infinite values.")
    model = bundle["pipeline"]
    pred = model.predict(X_in).astype(int)
    out = pd.DataFrame({
        "predicted_label": pred,
        "predicted_class": ["Seizure" if v == 1 else "Non-seizure" for v in pred],
        "model_name": bundle["model_name"],
        "prediction_timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "input_status": "valid"
    })
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_in)
        classes = list(model.classes_)
        if 1 in classes:
            out["seizure_probability"] = probabilities[:, classes.index(1)]
            out["score_interpretation"] = "classifier probability; calibration not independently established"
    elif hasattr(model, "decision_function"):
        out["decision_score"] = model.decision_function(X_in)
        out["score_interpretation"] = "decision score, not probability"
    return out

# Confirm the saved binary model reproduces the fitted model's predictions.
_binary_loaded = joblib.load(BINARY_MODEL_PATH)
assert np.array_equal(
    _binary_loaded["pipeline"].predict(X_test.iloc[:10]),
    best_binary_model.predict(X_test.iloc[:10])
)
print("Binary model serialization check passed.")


## 10. Raw EEG prediction

In [ ]:
# 10. Raw signal -> features -> predictions
def predict_raw_signal(signal, fs=FS, bundle=None):
    """
    Predict one Bonn-style full recording (not a continuous seizure event stream).
    Compatible only with a model trained from the raw-extracted feature schema.
    """
    if bundle is None:
        bundle = load_model_bundle()
    if bundle.get("dataset_mode") != "raw_extracted":
        raise ValueError(
            "This model was trained from precomputed features. Raw-signal prediction is disabled because "
            "the precomputed feature definitions may differ. Rebuild features from raw Bonn files and retrain."
        )
    if not np.isscalar(fs) or not np.isfinite(fs) or fs <= 0:
        raise ValueError("Sampling frequency must be a positive finite number.")
    if not np.isclose(fs, bundle["sampling_frequency_hz"], rtol=0, atol=1e-6):
        raise ValueError(f"Sampling frequency mismatch: expected {bundle['sampling_frequency_hz']} Hz, received {fs} Hz.")
    x = np.asarray(signal, dtype=float)
    if x.ndim != 1 or x.size < 1000:
        raise ValueError("Expected a valid full-length, one-dimensional Bonn EEG recording.")
    if not np.isfinite(x).all():
        raise ValueError("Raw EEG contains NaN or infinite values.")
    filtered = bandpass_filter(x, fs=fs)
    feature_row = extract_epoch_features(filtered, fs=fs)
    feature_frame = pd.DataFrame([feature_row])
    expected = list(bundle["feature_columns"])
    missing = [c for c in expected if c not in feature_frame.columns]
    if missing:
        raise ValueError(f"Raw feature extractor/model schema mismatch. Missing features: {missing}")
    feature_frame = feature_frame[expected]
    predictions = predict_feature_rows(feature_frame, bundle)
    predictions["recording_samples"] = int(len(x))
    predictions["recording_duration_seconds"] = float(len(x) / fs)
    predictions["sampling_frequency_hz"] = fs
    return predictions

print("Raw-signal inference function defined.")
print("Use predict_raw_signal(signal_1d, fs=173.61) only when the saved model's feature schema matches this extractor.")

## 11. Integrity tests

In [ ]:
# 11. Lightweight tests
def run_integrity_tests():
    # Split integrity
    assert not (set(groups_train) & set(groups_test)), "Recording leakage between train and test."
    assert np.isfinite(X_train.select_dtypes(include=[np.number]).to_numpy(dtype=float)[
        np.isfinite(X_train.select_dtypes(include=[np.number]).to_numpy(dtype=float))
    ]).all()

    # Prediction should reject missing feature columns.
    test_bundle = load_model_bundle()
    try:
        predict_feature_rows(pd.DataFrame({"not_a_feature": [1.0]}), test_bundle)
        raise AssertionError("Missing feature columns were not rejected.")
    except ValueError:
        pass

    # Prediction should reject NaN/Inf.
    bad = X_test.iloc[:1].copy()
    bad.loc[bad.index[0], feature_cols[0]] = np.nan
    try:
        predict_feature_rows(bad, test_bundle)
        raise AssertionError("NaN feature input was not rejected.")
    except ValueError:
        pass

    # Serialization consistency
    reloaded = joblib.load(MODEL_PATH)
    before = best_model.predict(X_test.iloc[:10])
    after = reloaded["pipeline"].predict(X_test.iloc[:10])
    assert np.array_equal(before, after), "Serialized pipeline predictions changed."

    # Raw-signal validation
    for bad_signal in [np.array([]), np.array([1.0, np.nan, 2.0])]:
        try:
            predict_raw_signal(bad_signal)
            raise AssertionError("Invalid raw signal was not rejected.")
        except (ValueError, FileNotFoundError):
            pass

    print("All integrity tests passed.")

run_integrity_tests()

## 12. Export results

In [ ]:
# 12. Final summary and export
summary = {
    "dataset_mode": DATA_SOURCE_MODE,
    "n_epochs": int(len(data)),
    "n_recordings": int(data[GROUP_COLUMN].nunique()),
    "n_features": int(len(feature_cols)),
    "classes": LABEL_NAMES,
    "train_epochs": int(len(train_idx)),
    "test_epochs": int(len(test_idx)),
    "train_recordings": int(len(set(groups_train))),
    "test_recordings": int(len(set(groups_test))),
    "group_overlap": int(len(set(groups_train) & set(groups_test))),
    "selected_model": best_name,
    "selection_metric": "grouped-CV macro F1; tie-breakers active-seizure sensitivity and test macro F1",
    "best_model_path": str(MODEL_PATH.resolve()),
    "output_directory": str(OUTPUT_DIR.resolve()),
    "limitations": [
        "No verified patient identifiers in supplied feature files; recording-wise split is not proof of patient-wise generalization.",
        "Precomputed-feature mode cannot reconstruct or validate upstream raw EEG filtering and normalization.",
        "Bonn dataset results do not establish clinical validity or continuous seizure-event detection performance.",
        "Probabilities are not guaranteed to be calibrated.",
        "Event-level sensitivity, false alarms per hour, and detection delay require timestamped seizure annotations and continuous recordings."
    ]
}
with open(OUTPUT_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nMain outputs:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(p.name)

## Interpretation and limitations

- Bonn is a small, curated benchmark; high scores do not establish real-world seizure detection.
- Recordings are split by file, not verified patient ID. Subject-independent generalization is not established.
- S vs N+F is a more focused comparison but still uses separate Bonn recording sets, not continuous monitoring.
- Model probability outputs are not necessarily calibrated.
- Clinical use would require independent datasets, patient-level evaluation, timestamped event annotations, false alarms per hour, and detection-delay analysis.
